# Code Review - Transformer
교재 : (파이토치 트랜스포머를 활용한) 자연어 처리와 컴퓨터 비전 심층학습  
범위 : 2.7장 트랜스포머 - Transformer


## Data

사용 데이터: Multi30k
- 영어-독일어 병렬 말뭉치(Parallel corpus)
- 약 30,000개

**Install libraries**

In [1]:
!pip install torchdata torchtext portalocker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.1 MB/s eta 0:00:00


* torchdata: 대규모 데이터셋을 쉽게 불러오고 변환 및 배치하는 API 제공
* torchtext: 텍스트 전처리 및 데이터셋 관리 단순화를 위한 라이브러리
* portalocker: 파이썬에서 파일 락을 관리하기 위한 라이브러리, 파일 락을 사용해 여러 프로세스 간에 동시에 파일을 수정하거나 읽는 것을 방지함 (Multi30k 다운로드 및 압축 해제 과정에서 내부적으로 사용됨)



---


(+) 버전 문제   
1. torchdata(0.18.0)가 기존 torch 버전(2.10.0)이랑 호환이 안 돼서 torch를 호환가능한 2.3.0으로 다운그레이드

> TorchText 0.18 (https://github.com/pytorch/text/releases)   
> Warning: TorchText development is stopped and the 0.18 release will be the last stable release of the library.
>
> This release is compatible with PyTorch 2.3.0 patch release.


2. Multi30k를 이용해서 데이터를 불러오려면 내부적으로 torchdata의 datapipes가 필요한데, 기존 torchdata 버전(0.11.0)에는 더 이상 사용되지 않아서 datapipes가 마지막으로 포함된 0.9.0 버전으로 다운그레이드



> TorchData 0.10.1 (https://meta-pytorch.org/data/0.10/index.html)   
> Removing DataPipes and DataLoader V2. We’ll also be revisiting the DataPipes > references in pytorch/pytorch. In release torchdata==0.8.0 (July 2024) they > will be marked as deprecated, and in 0.10.0 (Late 2024) they will be deleted.



-> deprecation_warning()이 발생하긴 하지만 우선은 코드 테스트를 위해 그대로 사용

In [2]:
!pip uninstall torch torchdata
!pip install torch==2.3.0, torchdata==0.9.0

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Would remove:
    /usr/local/bin/torchfrtrace
    /usr/local/bin/torchrun
    /usr/local/lib/python3.12/dist-packages/functorch/*
    /usr/local/lib/python3.12/dist-packages/torch-2.10.0+cu128.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torch/*
    /usr/local/lib/python3.12/dist-packages/torchgen/*
Proceed (Y/n)? Y
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchdata 0.11.0
Uninstalling torchdata-0.11.0:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/torchdata-0.11.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/torchdata/*
Proceed (Y/n)? Y
  Successfully uninstalled torchdata-0.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [3]:
import torch
import torchdata
import torchtext
import warnings
warnings.filterwarnings('ignore')

print("torch 버전:", torch.__version__)
print("torchdata 버전:", torchdata.__version__)
print("torchtext 버전:", torchtext.__version__)

torch 버전: 2.3.0+cu121
torchdata 버전: 0.9.0+cpu
torchtext 버전: 0.18.0+cpu


**Download & Preprocess**

In [5]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 111.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
from torchtext.datasets import Multi30k
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

def generate_tokens(text_iter, language):
    """ 토큰 리스트를 생성하는 generator"""
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1} # 각 튜플 내 언어 위치

    for text in text_iter:
        yield token_transform[language](text[language_index[language]]) # yield: 요청 시 값을 함수 밖으로 전달


SRC_LANGUAGE = "de" # Source 언어: 독일어
TGT_LANGUAGE = "en" # Target 언어: 영어
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3 # 특수 토큰 인덱스
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]  # 특수 토큰

### 언어별 tokenizer 지정: spaCy 라이브러리의 사전 학습된 모델을 지정
token_transform = {
    SRC_LANGUAGE: get_tokenizer("spacy", language="de_core_news_sm"),
    TGT_LANGUAGE: get_tokenizer("spacy", language="en_core_web_sm"),
}
print("Token Transform:")
print(token_transform)

### 언어별 어휘 사전 생성: 토큰을 정수로 변환하는 역할
vocab_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    train_iter = Multi30k(split="train", language_pair=(SRC_LANGUAGE, TGT_LANGUAGE)) # Multi30k 데이터셋 iterator -> (독일어, 영어) 튜플 형식

    vocab_transform[language] = build_vocab_from_iterator(
        generate_tokens(train_iter, language),  # 토큰 generator
        min_freq=1,                 # 토큰화된 단어들의 최소 빈도수
        specials=special_symbols,   # 특수 토큰 지정
        special_first=True,         # 특수 토큰을 단어 집합 맨 앞에 추가
    )

# 어휘 사전에 없는 토큰은 <unk> 인덱스 할당
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    vocab_transform[language].set_default_index(UNK_IDX)

print("Vocab Transform:")
print(vocab_transform)

Token Transform:
{'de': functools.partial(<function _spacy_tokenize at 0x79c18c1e1440>, spacy=<spacy.lang.de.German object at 0x79c18650eff0>), 'en': functools.partial(<function _spacy_tokenize at 0x79c18c1e1440>, spacy=<spacy.lang.en.English object at 0x79c1701bc680>)}
Vocab Transform:
{'de': Vocab(), 'en': Vocab()}


# Model

**Model class**

In [7]:
import math
import torch
from torch import nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        """ PositionalEncoding: 임베딩 벡터에 위치 정보를 추가
        d_model: 임베딩 크기
        max_len: 최대 시퀀스 길이
        dropout: dropout 비율
        """
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1) # 시퀀스 위치
        div_term = torch.exp(                         # 임베딩 차원마다 서로 다른 주파수 생성(짝수 차원 기준으로 (d_model/2)개의 주파수를 정의하고, 하나의 주파수를 sin, cos 쌍으로 사용)
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        ### 시퀀스 위치와 임베딩 차원에 따라 다른 값을 가지는 pe 텐서 생성
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term) # 짝수 차원: sin
        pe[:, 0, 1::2] = torch.cos(position * div_term) # 홀수 차원: cos
        self.register_buffer("pe", pe)  # 학습하지 않는 텐서로 등록

    def forward(self, x):
        ### 입력 임베딩 + 위치 인코딩
        x = x + self.pe[: x.size(0)] # pe에서 입력 시퀀스 길이만큼만 선택
        return self.dropout(x)


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        """ TokenEmbedding: 토큰을 임베딩 벡터로 변환"""
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size) # 어휘 사전 크기를 입력받아 임베딩 크기로 변환
        self.emb_size = emb_size

    def forward(self, tokens):
        ### 토큰을 임베딩 벡터로 변환 후 스케일링
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size) # 임베딩 크기가 커질수록 분산이 커지는 것을 방지


class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        max_len,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward,
        dropout=0.1,
    ):
        """ Seq2SeqTransformer: Encoder-Decoder 구조의 Transformer 모델 """
        super().__init__()
        ### 언어별 토큰 임베딩
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        ### 위치 인코딩
        self.positional_encoding = PositionalEncoding(
            d_model=emb_size, max_len=max_len, dropout=dropout
        )
        ### Transformer 모델
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )
        ### Decoder에서 출력된 임베딩 벡터를 target 어휘 사전에 대한 logit으로 선형 변환
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(
        self,
        src,
        tgt,
        src_mask,
        tgt_mask,
        src_padding_mask,
        tgt_padding_mask,
        memory_key_padding_mask,
    ):
        ### 1. TokenEmbedding + PositionalEncoding
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt))
        ### 2. Transformer
        outs = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        ### 3. Target vocabulary logit
        return self.generator(outs)

    def encode(self, src, src_mask):
        return self.transformer.encoder(
            self.positional_encoding(self.src_tok_emb(src)), src_mask
        )

    def decode(self, tgt, memory, tgt_mask):
        return self.transformer.decoder(
            self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask
        )

nn.Transformer()

In [8]:
transformer = nn.Transformer(
    d_model=512,                         # 입출력 차원의 크기(=임베딩 크기)
    nhead=8,                             # multi-head attention의 head 개수
    num_encoder_layers=6,                # 인코더/디코더 계층 수
    num_decoder_layers=6,
    dim_feedforward=2048,                # feed-forward layer의 은닉층 크기
    dropout=0.1,                         # 인코더/디코더 계층에 적용되는 dropout 비율
    activation=torch.nn.functional.relu, # feed-forward의 활성화 함수
    layer_norm_eps=1e-05                 # layernorm 시 분모에 더해지는 epsilon 값
    )

nn.Transformer.forward()

In [ ]:
output = transformer.forward(
    src,                          # 소스/타깃 시퀀스 [src/tgt_seq_len, batch, emb_size]
    tgt,
    src_mask=None,                # 소스/타깃 시퀀스의 마스크 [src/tgy_seq_len, src/tgt_seq_len]
    tgt_mask=None,
    src_padding_mask=None,        # 소스/타깃 패딩 마스크 [batch, src/tgt_seq_len]
    tgt_padding_mask=None,
    memory_key_padding_mask=None, # 인코더 출력의 마스크 [tgt_seq_len, src_seq_len]
    )

- src_mask/tgt_mask : self-attention에서 attention score에 적용되는 마스크로 각 가중치의 영향력을 조절
    - mask 값이 bool type인 경우 -> True이면 기존 가중치를 그대로 유지하고,False이면 해당 위치를 -inf로 채워서 어텐션을 차단한다.
    - mask 값이 bool type이 아닌 경우 -> mask 값을 attention score에 더한다. -inf 이면 softmax 값이 0이 되어 해당 위치의 정보를 모델이 무시하게 만든다. 반대로 +inf 이면 softmax 1이 되어 해당 위치의 정보만으로 구성된다.

    ```
    # https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html 참고
    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_bias.masked_fill_(attn_mask.logical_not(), float("-inf"))
        else:
            attn_bias = attn_mask + attn_bias
    ```
- src/tgt_padding_mask : 시퀀스에서 패딩 토큰에 적용하는 마스크로 패딩 토큰은 실제 의미를 가지지 않는 것으로 간주되어 가중치를 0으로 만든다.
- memory_key_padding_mask : 메모리 마스크의 값이 0인 위치에서는 어텐션 연산을 수행하지 않는다.

# Train

**Define model**

In [10]:
from torch import optim


BATCH_SIZE = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

### Model
model = Seq2SeqTransformer(
    num_encoder_layers=3,
    num_decoder_layers=3,
    emb_size=512,
    max_len=512,
    nhead=8,
    src_vocab_size=len(vocab_transform[SRC_LANGUAGE]),
    tgt_vocab_size=len(vocab_transform[TGT_LANGUAGE]),
    dim_feedforward=512,
).to(DEVICE)

### Loss function & Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX).to(DEVICE) # PAD_IDX는 모델 학습과 예측 시 무시
optimizer = optim.Adam(model.parameters())

### Architecture
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("│  │  └", sssub_name)

src_tok_emb
└ embedding
tgt_tok_emb
└ embedding
positional_encoding
└ dropout
transformer
└ encoder
│  └ layers
│  │  └ 0
│  │  └ 1
│  │  └ 2
│  └ norm
└ decoder
│  └ layers
│  │  └ 0
│  │  └ 1
│  │  └ 2
│  └ norm
generator


**DataLoader**

In [11]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence


def sequential_transforms(*transforms):
    """ sequential_transforms: 여러 개의 전처리 함수를 인자로 받아, 이를 차례로 적용하는 함수 func를 반환"""
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func

def input_transform(token_ids):
    """ input_transform: 문장의 시작(BOS_IDX)과 끝(EOS_IDX)을 나타내는 특수 토큰 인덱스를 문장 앞뒤에 concat"""
    return torch.cat(
        (torch.tensor([BOS_IDX]), torch.tensor(token_ids), torch.tensor([EOS_IDX]))
    )

def collator(batch):
    """ collator: 배치 단위로 데이터를 처리"""
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        # 문자열 끝 개행 문자(\n) 제거 + text 전처리 함수 순차 적용
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip("\n")))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n")))

    # 시퀀스들이 동일한 길이를 가지도록 시퀀스 뒤쪽에 패딩 토큰(PAD_IDX)을 추가
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch

### 언어별 text 전처리 함수 집합을 저장
text_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    text_transform[language] = sequential_transforms(
        token_transform[language], vocab_transform[language], input_transform
    ) # 토큰화, 인덱스화, 특수 토큰 인덱스 추가를 차례로 적용하는 함수 저장

### DataLoader 정의
data_iter = Multi30k(split="valid", language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)
source_tensor, target_tensor = next(iter(dataloader))

print("(source, target):")
# print(next(iter(data_iter))) # Exception: OnDiskCache 데이터 자동 다운로드를 시도했으나  300초(5분) 안에 완료되지 않아 타임아웃 발생

print("source_batch:", source_tensor.shape)
print(source_tensor)

print("target_batch:", target_tensor.shape)
print(target_tensor)

(source, target):
source_batch: torch.Size([35, 128])
tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [  14,    5,    5,  ...,    5,   21,    5],
        [  38,   12,   35,  ...,   12, 1750,   69],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])
target_batch: torch.Size([30, 128])
tensor([[   2,    2,    2,  ...,    2,    2,    2],
        [   6,    6,    6,  ...,  250,   19,    6],
        [  39,   12,   35,  ...,   12, 3254,   61],
        ...,
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1],
        [   1,    1,    1,  ...,    1,    1,    1]])


**Generate attention mask**

In [12]:
def generate_square_subsequent_mask(s):
    """ generate_square_subsequent_mask: s x s 크기의 mask 생성 (s: seq_len) """
    # ones(1로 이뤄진 sxs 크기의 행렬) -> triu(상삼각행렬) -> transpose(전치) => 하삼각행렬
    mask = (torch.triu(torch.ones((s, s), device=DEVICE)) == 1).transpose(0, 1)
    # Mask 값이 0이면 -inf, 1이면 0.0으로 채움
    mask = (
        mask.float()
        .masked_fill(mask == 0, float("-inf"))
        .masked_fill(mask == 1, float(0.0))
    )
    return mask


def create_mask(src, tgt):
    """ create_mask: src, tgt 데이터의 mask 생성"""
    ### 시퀀스 길이 계산
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]

    ### Mask 생성
    tgt_mask = generate_square_subsequent_mask(tgt_seq_len) # tgt mask는 상삼각 부분이 -inf으로 채워진 tgt_seq_len x tgt_seq_len 크기의 행렬 -> 예측하고자 하는 위치의 뒤에 오는 토큰은 참조 불가능
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=DEVICE).type(torch.bool) # src mask는 False로 채워진 src_seq_len x src_seq_len 크기의 행렬 -> 모든 토큰을 대상으로 셀프 어텐션 수행 가능

    ### Padding mask 생성
    src_padding_mask = (src == PAD_IDX).transpose(0, 1)
    tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask


target_input = target_tensor[:-1, :]
target_out = target_tensor[1:, :]

source_mask, target_mask, source_padding_mask, target_padding_mask = create_mask(
    source_tensor, target_input
)

print("source_mask:", source_mask.shape)
print(source_mask)
print("target_mask:", target_mask.shape)
print(target_mask)
print("source_padding_mask:", source_padding_mask.shape)
print(source_padding_mask)
print("target_padding_mask:", target_padding_mask.shape)
print(target_padding_mask)

source_mask: torch.Size([35, 35])
tensor([[False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        ...,
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False],
        [False, False, False,  ..., False, False, False]], device='cuda:0')
target_mask: torch.Size([29, 29])
tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf,
         -inf, -inf, -inf,

**Train & Validation**

In [13]:
def run(model, optimizer, criterion, split):
    model.train() if split == "train" else model.eval() # 학습/평가 모드 설정

    data_iter = Multi30k(split=split, language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)

    losses = 0
    for source_batch, target_batch in dataloader:
        source_batch = source_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        target_input = target_batch[:-1, :] # target input: <eos>를 제외한 target 시퀀스 -> decoder 입력
        target_output = target_batch[1:, :] # target output: <bos>를 제외한 target 시퀀스 -> 정답 라벨

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
            source_batch, target_input
        )   # src, tgt mask 생성

        logits = model(
            src=source_batch,
            tgt=target_input,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask,
        )

        optimizer.zero_grad()
        loss = criterion(logits.reshape(-1, logits.shape[-1]), target_output.reshape(-1))
        if split == "train": # 학습 시 -> 역전파 + 파라미터 업데이트
            loss.backward()
            optimizer.step()
        losses += loss.item()

    return losses / len(list(dataloader))


for epoch in range(5):
    train_loss = run(model, optimizer, criterion, "train")
    val_loss = run(model, optimizer, criterion, "valid")
    print(f"Epoch: {epoch+1}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}")

Epoch: 1, Train loss: 4.621, Val loss: 3.887
Epoch: 2, Train loss: 3.689, Val loss: 3.596
Epoch: 3, Train loss: 3.413, Val loss: 3.535
Epoch: 4, Train loss: 3.247, Val loss: 3.457
Epoch: 5, Train loss: 3.104, Val loss: 3.459


**Test**

In [14]:
def greedy_decode(model, source_tensor, source_mask, max_len, start_symbol):
    """ greedy_decode: decoder가 생성한 확률 분포에서 가장 높은 확률을 가지는 단어 선택 """
    source_tensor = source_tensor.to(DEVICE)
    source_mask = source_mask.to(DEVICE)

    ### Encoding
    memory = model.encode(source_tensor, source_mask) # source 문장을 인코딩하여 memory에 저장

    ### Decoding
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE) # target 데이터의 입력 텐서
    for i in range(max_len - 1):
        memory = memory.to(DEVICE)
        target_mask = generate_square_subsequent_mask(ys.size(0)) # 현재 target 시퀀스 길이에 맞는 mask 생성
        target_mask = target_mask.type(torch.bool).to(DEVICE)

        out = model.decode(ys, memory, target_mask) # target 시퀀스 디코딩 [tgt_seq_len, batch, emb_size]
        out = out.transpose(0, 1)                   # [batch, tgt_seq_len, emb_size]
        prob = model.generator(out[:, -1])          # 각 batch의 마지막 토큰을 인덱싱 후 vocab logit로 변환 [batch, emb_size]
        _, next_word = torch.max(prob, dim=1)       # 확률이 가장 높은 토큰의 인덱스를 next_word에 저장
        next_word = next_word.item()

        ys = torch.cat(
            [ys, torch.ones(1, 1).type_as(source_tensor.data).fill_(next_word)], dim=0
        ) # 예측한 토큰 인덱스 concat
        if next_word == EOS_IDX: # EOS_IDX가 예측되면 종료
            break
    return ys # 예측된 토큰 인덱스 시퀀스 반환


def translate(model, source_sentence):
    """ translate: source 문장을 학습된 model로 번역"""
    model.eval() # 평가 모드
    ### Source 문장 전처리
    source_tensor = text_transform[SRC_LANGUAGE](source_sentence).view(-1, 1) # src_seq_len x 1 크기로 변환
    num_tokens = source_tensor.shape[0] # src 토큰 개수
    src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)

    ### Target 토큰 인덱스 예측
    tgt_tokens = greedy_decode(
        model, source_tensor, src_mask, max_len=num_tokens + 5, start_symbol=BOS_IDX # max_len은 source 문장보다 약간 더 길어지는 경우가 많으므로 (source 토큰 수 + 5)로 설정
    ).flatten()

    ### Target vocab logit -> sentence
    output = vocab_transform[TGT_LANGUAGE].lookup_tokens(list(tgt_tokens.cpu().numpy()))[1:-1] # 어휘 사전의 lookup_tokens를 이용해 텍스트로 변환 후 bos, eos 제거
    return " ".join(output) # 단어 리스트를 공백으로 연결

output_oov = translate(model, "Eine Gruppe von Menschen steht vor einem Iglu .") # OOV(Out-of_Vocabulary) iglu 포함
output = translate(model, "Eine Gruppe von Menschen steht vor einem Gebäude .")
print(output_oov)
print(output)

A group of people are standing on a boat .
A group of people are standing on a boat .
